In [5]:
from qvarnet.models.exponential import LogExponentialMLPwithPenalty
from qvarnet.train import train
import jax
import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp
from qvarnet.probability import build_prob_fn
from qvarnet.sampling_step import sample_and_process
from qvarnet.training_step import energy_and_grads
from qvarnet.hamiltonian.continuous import HarmonicOscillatorHamiltonian

# SETUP

In [ ]:
N_PARTICLES = 2
DIM = 1
N_CHAINS = 5_000
DoF = N_PARTICLES * DIM
SHAPE = (N_CHAINS, DoF)
EPOCHS = 1000
N_STEPS = 10

model = LogExponentialMLPwithPenalty(
        architecture=[N_PARTICLES, 128, 1],
        hidden_activation=jax.nn.tanh,
        kernel_init=jax.nn.initializers.normal(stddev=.01),
        bias_init=jax.nn.initializers.normal(stddev=.01),
    )

In [7]:
def _cm_relative(x, n_particles, n_dim):
    """Subtract center-of-mass from particle coordinates.

    x:      (..., n_particles * n_dim)
    r:      (..., n_particles, n_dim)
    cm:     (..., 1, n_dim)
    return: (..., n_particles * n_dim)  — same shape as input, CM-subtracted
    """
    shape = x.shape[:-1]
    r = x.reshape(*shape, n_particles, n_dim)  # (..., n_particles, n_dim)
    cm = r.mean(axis=-2, keepdims=True)        # (..., 1, n_dim)
    return (r - cm).reshape(*shape, n_particles * n_dim)

In [ ]:
def compute_with_transform_in_model():
    def model_apply(params, x_batch):
        return model.apply(params, _cm_relative(x_batch, n_particles=N_PARTICLES, n_dim=DIM))
    
    params = model.init(jax.random.PRNGKey(0), jnp.zeros(SHAPE))  # Initialize model parameters

    for epoch in range(EPOCHS):
        # Sample a batch of configurations
        x_batch = sample_and_process(key=jax.random.PRNGKey(epoch),
                                    prob_fn=build_prob_fn(model_apply, True),
                                    prob_params=params,
                                    init_positions=jnp.zeros(SHAPE),
                                    step_size=0.1,
                                    n_chains=N_CHAINS,
                                    DoF=DoF,
                                    n_steps=N_STEPS,
                                    burn_in=N_STEPS-1,
                                    thinning=1,
                                    PBC=1.0,
                                    is_log_prob=True)
        print(f"Sampled batch shape: {x_batch.shape}")  # Debugging statement to check the shape of x_batch
        
        # Compute the loss and update the model parameters
        energy, s_energy, grads = energy_and_grads(hamiltonian=HarmonicOscillatorHamiltonian(),
                                                    params=params,
                                                    batch=x_batch,
                                                    model_apply=model_apply,
                                                    is_log_model=True)
        
        params = jax.tree_multimap(lambda p, g: p - 0.01 * g, params, grads)  # Simple SGD update

        # Optionally, you can log the training progress
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Energy = {energy}, Grad Norm = {jax.tree_util.tree_reduce(lambda x, y: x + jnp.sum(jnp.square(y)), 0, grads)}")

def compute_outside_model():
    def model_apply(params, x_batch):
        return model.apply(params, x_batch)

    for epoch in range(EPOCHS):
        # Sample a batch of configurations
        x_batch = sample_and_process(key=jax.random.PRNGKey(epoch),
                                    prob_fn=build_prob_fn(model_apply, True),
                                    prob_params=params,
                                    init_positions=jnp.zeros(SHAPE),
                                    step_size=0.1,
                                    n_chains=N_CHAINS,
                                    DoF=DoF,
                                    n_steps=N_STEPS,
                                    burn_in=N_STEPS-1,
                                    thinning=1,
                                    PBC=1.0,
                                    is_log_prob=True)

        x_batch = _cm_relative(x_batch, n_particles=N_PARTICLES, n_dim=DIM)  # Apply CM-relative transform to the batch
        
        # Compute the loss and update the model parameters
        energy, s_energy, grads = energy_and_grads(hamiltonian=HarmonicOscillatorHamiltonian(),
                                                    params=params,
                                                    batch=x_batch,
                                                    model_apply=model_apply,
                                                    is_log_model=True)
        
        params = jax.tree_multimap(lambda p, g: p - 0.01 * g, params, grads)  # Simple SGD update

        # Optionally, you can log the training progress
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Energy = {energy}, Grad Norm = {jax.tree_util.tree_reduce(lambda x, y: x + jnp.sum(jnp.square(y)), 0, grads)}")

In [15]:
compute_with_transform_in_model()

ValueError: vmap got inconsistent sizes for array axes to be mapped:
  * one axis had size 1: axis 0 of argument random_values of type float32[1,100,4];
  * one axis had size 5000: axis 0 of argument init_position of type float32[5000,2]